# 9단계: Hugging Face 업로드 모델 평가

목표:
- Hugging Face Hub에 업로드된 LoRA adapter와 merged model을 확인한다.
- 테스트 입력 5개에 대해 poem, 줄 수, validation_reason을 출력한다.
- 검증 실패해도 출력을 숨기지 않고 그대로 보여준다.


## 1. 설치


In [ ]:
!pip -q install -U "transformers>=4.47.0" "accelerate>=1.2.0" "peft>=0.14.0" "torchao>=0.16.0" bitsandbytes huggingface_hub sentencepiece


## 2. 설정

`MODEL_KIND`를 `merged` 또는 `adapter`로 바꿔서 각각 테스트할 수 있다.


In [ ]:
import re
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_REPO_ID = "z-unghyun/poem-generator-lora-adapter"
MERGED_REPO_ID = "z-unghyun/poem-generator-merged"

MODEL_KIND = "merged"  # "merged" or "adapter"

TEST_INPUTS = [
    "야자 끝나고 비 오는 정류장에서 버스를 기다림",
    "새벽에 과제를 하다가 모니터 빛에 눈이 아픔",
    "카페에서 창업 아이디어를 정리하다가 갑자기 막힘",
    "혼자 여행지 술집에서 감자튀김과 맥주를 먹음",
    "AI 프로젝트를 하다가 언어가 망가지는 걸 봄",
]

print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 3. 모델 로드

CPU에서도 동작하게 구성했지만, Colab GPU가 있으면 훨씬 빠르다.


In [ ]:
def load_merged_model():
    tokenizer = AutoTokenizer.from_pretrained(MERGED_REPO_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto",
    }
    if torch.cuda.is_available():
        load_kwargs["dtype"] = torch.float16
    else:
        load_kwargs["dtype"] = torch.float32

    model = AutoModelForCausalLM.from_pretrained(MERGED_REPO_ID, **load_kwargs)
    model.eval()
    return tokenizer, model

def load_adapter_model():
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            dtype=torch.float32,
            device_map="auto",
            trust_remote_code=True,
        )

    model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)
    model.eval()
    return tokenizer, model

started = time.time()
if MODEL_KIND == "merged":
    tokenizer, model = load_merged_model()
elif MODEL_KIND == "adapter":
    tokenizer, model = load_adapter_model()
else:
    raise ValueError("MODEL_KIND must be 'merged' or 'adapter'")

print("loaded_model_kind:", MODEL_KIND)
print("elapsed_seconds:", round(time.time() - started, 2))


## 4. 검증 함수

실패 출력도 숨기지 않고 `invalid_shown`으로 표시한다.


In [ ]:
SYSTEM_PROMPT = "너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다. 설명 없이 시만 정확히 3줄로 쓴다."
PROSE_PATTERNS = ("입니다", "합니다", "것이다", "설명한다", "의미한다", "중요하다")

def line_count(text: str) -> int:
    return len([line for line in text.splitlines() if line.strip()])

def validation_reason(poem: str) -> str:
    lines = [line.strip() for line in poem.splitlines() if line.strip()]
    if len(lines) != 3:
        return f"not_three_lines:{len(lines)}"
    if any(len(line) > 55 for line in lines):
        return "line_too_long"
    if sum(poem.count(pattern) for pattern in PROSE_PATTERNS) >= 2:
        return "prose_report_like_output"
    if re.search(r"[A-Za-z]{12,}", poem):
        return "broken_korean_like_output"
    return "ok"

def validation_status(reason: str) -> str:
    return "valid" if reason == "ok" else "invalid_shown"

def build_prompt(experience: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"경험: {experience}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_poem(experience: str) -> str:
    prompt = build_prompt(experience)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=70,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


## 5. 5개 테스트 실행


In [ ]:
results = []
for experience in TEST_INPUTS:
    started = time.time()
    poem = generate_poem(experience)
    reason = validation_reason(poem)
    result = {
        "experience": experience,
        "poem": poem,
        "line_count": line_count(poem),
        "validation_status": validation_status(reason),
        "validation_reason": reason,
        "elapsed_seconds": round(time.time() - started, 3),
    }
    results.append(result)

for result in results:
    print("=" * 80)
    print("경험:", result["experience"])
    print("시:")
    print(result["poem"])
    print("line_count:", result["line_count"])
    print("validation_status:", result["validation_status"])
    print("validation_reason:", result["validation_reason"])
    print("elapsed_seconds:", result["elapsed_seconds"])


## 6. 요약


In [ ]:
valid_count = sum(1 for r in results if r["validation_status"] == "valid")
print("model_kind:", MODEL_KIND)
print("adapter_repo:", ADAPTER_REPO_ID)
print("merged_repo:", MERGED_REPO_ID)
print("valid_count:", valid_count)
print("total_count:", len(results))
print("pass_rate:", f"{valid_count / len(results):.1%}")
